In [9]:
import numpy as np
import seaborn as sns
from numba import njit, prange
from typing import Callable, Dict
import inspect

import sys
sys.path.append('../')

import superstats as sup

## Specify Joint Prior

In [10]:
prior = sup.prior.JointPrior(
    v=sup.transition.AutoRegressionDrift(
        bounds=(0.0, 6.0),
        initial_prior=sup.prior.Prior(dist="normal", a=0.0, b=2.0),
        phi_prior=sup.prior.Prior(dist="beta", a=10.0, b=1.0),
        delta_prior=sup.prior.Prior(dist="normal", loc=0.0, scale=0.05),
        sigma_prior=sup.prior.Prior(dist="halfnormal", scale=0.1)
    ),
    a=sup.transition.AutoRegression(bounds=(0.5, 6.0)),
    tau=sup.transition.RandomWalk(bounds=(0.0, 2.0), sigma_prior=0.05),
    bias=sup.prior.Prior(dist="beta", a=25.0, b=25.0),
)

In [11]:
batch_size = 32
num_steps = 100
prior_draws = prior.sample(batch_size, num_steps)

## Specify Generative Model

In [12]:
cognitive_model = sup.simulation.sample_ddm

In [13]:
generative_model = sup.simulation.GenerativeModel(
    prior=prior,
    model=cognitive_model,
)

In [16]:
%%timeit
sim_data = generative_model.sample(batch_size=32, steps=500)

48.3 ms ± 2.59 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [18]:
sim_data = generative_model.sample(batch_size=32, steps=500)
sim_data.keys()

dict_keys(['data', 'local_params', 'global_params', 'shared_params'])